# Paid to Disagree: combine evaluation seeds into the paper tables (CPU)

Collects every `runs/<prefix>_seed<k>/` folder from this notebook's inputs, recomputes the
metrics with the current code (so newer aggregators such as ISP and OW are included), combines
the seeds (mean, 95% interval across seeds, widest per-seed bootstrap interval), and writes the
paper tables and figures. **Needs no GPU:** set Accelerator to None.

**Inputs:** Add Input -> Your Work -> Notebooks, and add the notebook (and version) of every
seed run, e.g. `ptd-baseline-eval` (seed 0), `ptd-baseline-eval-s1`, `ptd-baseline-eval-s2`.
Then Save Version -> Save & Run All (Commit) and download `combined_tables.zip` from Output.

In [ ]:
# ---- settings ----
REPO = "https://github.com/nithish-74/paid-to-disagree.git"
RUN_PREFIX = "eval_untrained"       # matches RUN_PREFIX in the evaluation notebook
N_BOOT = 1000
OW_CALIBRATION = None               # optional: path to a dev-split run's metrics/ow_calibration.json

## 1. Code

In [ ]:
import os, subprocess, glob, pathlib, shutil, json
os.chdir("/kaggle/working")
if not pathlib.Path("ptd").exists():
    subprocess.run(["git", "clone", "-q", REPO, "ptd"], check=True)
os.chdir("/kaggle/working/ptd")
subprocess.run(["git", "pull", "-q", "--ff-only"], check=True)
print(subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)
!pip install -q -e ".[analysis]"

## 2. Collect the seed runs from the inputs

In [ ]:
found = sorted({str(pathlib.Path(p).parent) for p in glob.glob(f"/kaggle/input/**/{RUN_PREFIX}_seed*/members.jsonl", recursive=True)})
runs = {}
for src in found:
    name = pathlib.Path(src).name
    if not pathlib.Path(src, "questions.jsonl").exists():
        print(f"skip {src}: no questions.jsonl (that seed did not finish)")
        continue
    if name in runs:
        print(f"skip {src}: {name} already taken from {runs[name]}")
        continue
    runs[name] = src
    shutil.copytree(src, f"runs/{name}", dirs_exist_ok=True)
print("seed runs:", sorted(runs))
assert runs, "no finished seed runs found: add the evaluation notebooks' outputs as inputs"

## 3. Metrics per seed (current code), then combine and build the tables

In [ ]:
cal = ["--ow-calibration", OW_CALIBRATION] if OW_CALIBRATION else []
for name in sorted(runs):
    subprocess.run(["python", "-m", "eval.metrics", f"runs/{name}", "--n-boot", str(N_BOOT), *cal], check=True)
combined, paper = f"runs/{RUN_PREFIX}_combined", f"runs/{RUN_PREFIX}_paper"
subprocess.run(["python", "-m", "analysis.combine_seeds", *[f"runs/{n}" for n in sorted(runs)], "--out", combined], check=True)
subprocess.run(["python", "-m", "analysis.paper_tables", combined, "--out", paper], check=True)
print(open(f"{paper}/tables.md").read()[:6000])

## 4. Headline numbers and package

In [ ]:
print(json.dumps(json.load(open(f"{paper}/summary.json")), indent=1)[:4000])
pack = pathlib.Path("/kaggle/working/combined_tables")   # small: tables, figures, per-seed metrics
shutil.rmtree(pack, ignore_errors=True)
shutil.copytree(combined, pack / "combined")
shutil.copytree(paper, pack / "paper")
for name in sorted(runs):
    shutil.copytree(f"runs/{name}/metrics", pack / name / "metrics")
out = shutil.make_archive(str(pack), "zip", pack)
print(out, f"{os.path.getsize(out) / 1e6:.1f} MB")